In [1]:
import json
import pandas as pd
from vllm import LLM, SamplingParams
import evaluate

# Загрузка данных
val_holdout = []
with open("data/raw/jsonl/validation.jsonl", encoding="utf-8") as f:
    for line in f:
        val_holdout.append(json.loads(line))
val_df = pd.DataFrame(val_holdout)
print(f"val_holdout: {len(val_df)} rows")

ModuleNotFoundError: No module named 'vllm'

In [ ]:
# Инициализация модели через vLLM
# gpu_memory_utilization=0.7 — оставляем запас для системы
llm = LLM(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    gpu_memory_utilization=0.7,
    max_model_len=2048,
)

In [ ]:
# Промпт для суммаризации
PROMPT_TEMPLATE = (
    "Summarize the following dialogue in 1-2 sentences. "
    "Be concise and capture the main points.\n\n"
    "Dialogue:\n{dialogue}\n\nSummary:"
)

# Формируем промпты для val_holdout
prompts = [
    PROMPT_TEMPLATE.format(dialogue=d)
    for d in val_df["dialogue"].tolist()
]

# Параметры генерации: greedy, максимум 80 токенов
sampling_params = SamplingParams(temperature=0.0, max_tokens=80)

In [ ]:
# Генерация
outputs = llm.generate(prompts, sampling_params)

# Извлекаем текст ответа
predictions = [o.outputs[0].text.strip() for o in outputs]

# Посмотрим на пару примеров глазами
for i in range(2):
    print("=" * 100)
    print("DIALOGUE (first 300 chars):", val_df["dialogue"].iloc[i][:300])
    print("-" * 100)
    print("REFERENCE:", val_df["summary"].iloc[i])
    print("PREDICTED:", predictions[i])
    print()

# Метрики ROUGE и BLEU
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

refs = val_df["summary"].tolist()

rouge_scores = rouge.compute(predictions=predictions, references=refs, use_stemmer=True)
bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in refs])

print("\n=== LLM Baseline (Qwen2.5-0.5B-Instruct) on val_holdout ===")
print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")
print(f"BLEU:    {bleu_score['bleu']:.4f}")